In [1]:
import pandas as pd
import numpy as np
import altair as alt

# Data Cleaning and Exploratory Analysis: BOLT UBC First Byte

This notebook documents the cleaning and inspection of three datasets—Merchandise Sales, Fanbase Engagement, and Stadium Operations—for downstream analysis. The goal is to ensure consistency, correct estimated fields, and prepare features for segmentation and revenue modeling.


## Data Import and Initial Inspection
I begin by loading the raw datasets and inspecting their structure to identify missing values, data types, and potential inconsistencies.


In [2]:
merch = pd.read_csv('data/raw/BOLT UBC First Byte - Merchandise Sales.csv')

fanbase = pd.read_csv('data/raw/BOLT UBC First Byte - Fanbase Engagement.csv')

stadium = pd.read_csv('data/raw/BOLT UBC First Byte - Stadium Operations.csv')

In [3]:
#function for initial inspection of the three datasets

def inspect_data(df):
    head = df.head()
    describe = df.describe()
    info = df.info()

    return(head, describe, info)

In [4]:
inspect_data(stadium)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Month    144 non-null    int64 
 1   Source   144 non-null    object
 2   Revenue  144 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 3.5+ KB


(   Month      Source  Revenue
 0      1  Upper Bowl    25020
 1      2  Upper Bowl   686664
 2      3  Upper Bowl   434112
 3      4  Upper Bowl   390879
 4      5  Upper Bowl   286524,
             Month       Revenue
 count  144.000000  1.440000e+02
 mean     6.500000  9.189942e+04
 std      3.464102  1.341148e+06
 min      1.000000 -3.527500e+06
 25%      3.750000 -1.751925e+05
 50%      6.500000  1.420000e+05
 75%      9.250000  4.378755e+05
 max     12.000000  3.410190e+06,
 None)

In [5]:
#ensure date type columns are in datetime before inspection
merch['Arrival_Date'] = pd.to_datetime(merch['Arrival_Date'], errors='coerce')
merch['Selling_Date'] = pd.to_datetime(merch['Selling_Date'], errors='coerce')

In [6]:
inspect_data(merch)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67877 entries, 0 to 67876
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Product_ID          67877 non-null  int64         
 1   Barcode             67877 non-null  int64         
 2   Item_Category       67877 non-null  object        
 3   Item_Name           67877 non-null  object        
 4   Size                52041 non-null  object        
 5   Unit_Price          67877 non-null  int64         
 6   Customer_Age_Group  67877 non-null  object        
 7   Customer_Region     67877 non-null  object        
 8   Promotion           67877 non-null  bool          
 9   Channel             67877 non-null  object        
 10  Selling_Date        67877 non-null  datetime64[ns]
 11  Member_ID           67877 non-null  int64         
 12  Arrival_Date        54302 non-null  datetime64[ns]
dtypes: bool(1), datetime64[ns](2), int64(4), objec

(   Product_ID   Barcode Item_Category    Item_Name Size  Unit_Price  \
 0    10000000  20000000           Cap  Classic Cap  NaN          35   
 1    10000000  20000001           Cap  Classic Cap  NaN          35   
 2    10000000  20000002           Cap  Classic Cap  NaN          35   
 3    10000000  20000003           Cap  Classic Cap  NaN          35   
 4    10000000  20000004           Cap  Classic Cap  NaN          35   
 
   Customer_Age_Group Customer_Region  Promotion     Channel  \
 0              18–25        Domestic      False      Online   
 1              26–40        Domestic      False  Team Store   
 2              18–25        Domestic      False  Team Store   
 3                <18        Domestic       True      Online   
 4                <18   International      False  Team Store   
 
          Selling_Date  Member_ID        Arrival_Date  
 0 2024-02-14 09:30:00   10008632 2024-02-22 13:49:00  
 1 2024-01-31 14:30:00   10005467                 NaT  
 2 2024-04-3

In [7]:
inspect_data(fanbase)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Membership_ID    70000 non-null  int64 
 1   Age_Group        70000 non-null  object
 2   Games_Attended   70000 non-null  int64 
 3   Seasonal_Pass    70000 non-null  bool  
 4   Customer_Region  70000 non-null  object
dtypes: bool(1), int64(2), object(2)
memory usage: 2.2+ MB


(   Membership_ID Age_Group  Games_Attended  Seasonal_Pass Customer_Region
 0       10218708     18-25               3          False          Canada
 1       10043567     18-25               2          False          Canada
 2       14272475     41-60               2          False          Canada
 3       10034985     18-25               5          False          Canada
 4       10004699     18-25               7          False          Canada,
        Membership_ID  Games_Attended
 count   7.000000e+04    70000.000000
 mean    2.464124e+07        5.711414
 std     2.595011e+07        5.168150
 min     1.370000e+03        1.000000
 25%     1.001510e+07        3.000000
 50%     1.003260e+07        5.000000
 75%     3.167822e+07        7.000000
 max     9.999924e+07       30.000000,
 None)

In [8]:
for column in merch.columns:
    print(f"Unique values in '{column}': {merch[column].unique()}")

Unique values in 'Product_ID': [10000000 10000001 10000002 10000003 10000004 10000005 10000006 10000007
 10000008 10000009 10000010 10000011 10000012 10000013 10000014 10000015
 10000016 10000017]
Unique values in 'Barcode': [20000000 20000001 20000002 ... 20067874 20067875 20067876]


Unique values in 'Item_Category': ['Cap' 'Hoodie' 'Jersey' 'Mug' 'Poster' 'Scarf' 'Youth Hoodie'
 'Youth Jersey']
Unique values in 'Item_Name': ['Classic Cap' 'Retro Cap' 'Team Hoodie' 'Training Hoodie' 'Away Jersey'
 'Home Jersey' 'Retro Jersey' 'Club Mug' 'Logo Mug' 'Player Poster'
 'Team Poster' 'Away Scarf' 'Home Scarf' 'Youth Team Hoodie'
 'Youth Training Hoodie' 'Youth Away Jersey' 'Youth Home Jersey'
 'Youth Retro Jersey']
Unique values in 'Size': [nan 'L' 'M' 'S' 'XL' 'XXL' 'YL' 'YS' 'YM']
Unique values in 'Unit_Price': [ 35  75 152  25  20  40  50  90]
Unique values in 'Customer_Age_Group': ['18–25' '26–40' '<18' '60+' '41–60']
Unique values in 'Customer_Region': ['Domestic' 'International']
Unique values in 'Promotion': [False  True]
Unique values in 'Channel': ['Online' 'Team Store']
Unique values in 'Selling_Date': <DatetimeArray>
['2024-02-14 09:30:00', '2024-01-31 14:30:00', '2024-04-30 18:57:00',
 '2024-02-01 18:25:00', '2024-11-24 14:07:00', '2024-10-02 14:32:00',
 '202

In [9]:
for column in fanbase.columns:
    print(f"Unique values in '{column}': {fanbase[column].unique()}")

Unique values in 'Membership_ID': [10218708 10043567 14272475 ... 43388235 22532180 10035894]
Unique values in 'Age_Group': ['18-25' '41-60' '<18' '26-40' '60+']
Unique values in 'Games_Attended': [ 3  2  5  7  8 30  6 19  4 28  1 18 25 21 24 17 22 16 27 26 23 29 20 15]
Unique values in 'Seasonal_Pass': [False  True]
Unique values in 'Customer_Region': ['Canada' 'USA' 'Japan' 'India' 'China' 'South Korea' 'UK']


In [10]:
for column in stadium.columns:
    print(f"Unique values in '{column}': {stadium[column].unique()}")

Unique values in 'Month': [ 1  2  3  4  5  6  7  8  9 10 11 12]
Unique values in 'Source': ['Upper Bowl' 'Lower Bowl' 'Season' 'Premium' 'Food' 'Staff' 'Concert'
 'Conference' 'Insurance' 'Maintenance' 'Utilities' 'Advertising']
Unique values in 'Revenue': [   25020   686664   434112   390879   286524   187056   307230   462672
   358527   658944    58077        0  1154980  3327450  3204408  2628288
  2122992  1289960  1778832  2945364  2249916  3410190   556924   527340
   535923   478674   438702   228874   289464   480942   411540   416043
    35145   398400   272160   243033   169320   107856   183534   318816
   259521   314427    19965   443982  2777643  2596206  2227200  1723476
  1208020  1469214  2357364  2001792  2844126   299296 -2765000 -3400000
 -3318000 -3362000 -3361500 -3198000 -3485000 -3527500 -3276000 -3483000
 -3108000   450000   285000   620000  1200000   420000   720000  1180000
   240000    75000    70000   146000    71000    69000   144000    68500
   140000   2

## Checking for Duplicates in ID columns

In [11]:
has_duplicates = merch['Member_ID'].duplicated().any()

has_duplicates

np.True_

In [12]:
has_duplicates = merch['Product_ID'].duplicated().any()

has_duplicates

np.True_

In [13]:
has_duplicates = fanbase['Membership_ID'].duplicated().any()

has_duplicates

np.False_

Assumed purchase data behaviour: Each product purchase is represented as an instance, so a member ID may show up multiple times if they purchased multiple items. A particular combination of a membership ID and product ID may show up together if someone purchased an item multiple times - this would explain the data patterns observed in the merch dataset.

## Cleaning Merchandise Data

Observations
- `Customer_Age_Group` in merch is estimated and inconsistent.
- `Customer_Region` may reflect shipping destinations rather than true customer location.

In [14]:
merch_sorted = merch.sort_values(by = 'Member_ID')

merch_sorted.head()

,Product_ID,Barcode,Item_Category,Item_Name,Size,Unit_Price,Customer_Age_Group,Customer_Region,Promotion,Channel,Selling_Date,Member_ID,Arrival_Date
53292,10000004,20053292,Jersey,Away Jersey,L,152,<18,Domestic,True,Online,2023-12-01 03:15:00,10000000,2023-12-10 09:03:00
8437,10000007,20008437,Mug,Club Mug,NaN,25,26–40,Domestic,False,Online,2023-12-01 22:47:00,10000000,2023-12-09 13:10:00
12491,10000003,20012491,Hoodie,Training Hoodie,M,75,26–40,Domestic,False,Online,2023-12-01 05:45:00,10000001,2023-12-08 10:20:00
52572,10000014,20052572,Youth Hoodie,Youth Training Hoodie,YL,50,18–25,Domestic,True,Online,2023-12-01 01:35:00,10000002,2023-12-08 15:18:00
25608,10000011,20025608,Scarf,Away Scarf,NaN,40,18–25,Domestic,False,Online,2023-12-01 17:12:00,10000003,2023-12-08 11:38:00


The same membership ID will have different ages and regions belonging to it

In [15]:
fanbase = fanbase.rename(columns = {'Membership_ID':'Member_ID', 'Age_Group':'Customer_Age_Group'})

fanbase

,Member_ID,Customer_Age_Group,Games_Attended,Seasonal_Pass,Customer_Region
0,10218708,18-25,3,False,Canada
1,10043567,18-25,2,False,Canada
2,14272475,41-60,2,False,Canada
3,10034985,18-25,5,False,Canada
4,10004699,18-25,7,False,Canada
...,...,...,...,...,...
69995,10002392,<18,7,False,Canada
69996,90870179,18-25,3,False,Canada
69997,43388235,18-25,1,False,Canada
69998,22532180,18-25,8,False,Canada


In [16]:
fanbase['Customer_Age_Group'] = fanbase['Customer_Age_Group'].str.replace('-', '–')

In [17]:
fanbase.head()

,Member_ID,Customer_Age_Group,Games_Attended,Seasonal_Pass,Customer_Region
0,10218708,18–25,3,False,Canada
1,10043567,18–25,2,False,Canada
2,14272475,41–60,2,False,Canada
3,10034985,18–25,5,False,Canada
4,10004699,18–25,7,False,Canada


In [18]:
merged_df = pd.merge(merch, fanbase, on='Member_ID', how='outer', suffixes=('_merch', '_fanbase'), indicator=True)

#merged_df = merged_df[['Member_ID', 'Customer_Age_Group_fanbase','Customer_Age_Group_merch', 'Customer_Region_fanbase', 'Customer_Region_merch', 'Channel']]

merged_df.head()

,Product_ID,Barcode,Item_Category,Item_Name,Size,Unit_Price,Customer_Age_Group_merch,Customer_Region_merch,Promotion,Channel,Selling_Date,Member_ID,Arrival_Date,Customer_Age_Group_fanbase,Games_Attended,Seasonal_Pass,Customer_Region_fanbase,_merge
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,1370,NaT,26–40,4,False,Canada,right_only
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,7913,NaT,26–40,7,False,Canada,right_only
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,9041,NaT,18–25,4,False,Canada,right_only
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,13517,NaT,60+,4,False,Canada,right_only
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,18474,NaT,<18,4,False,India,right_only


Use the fanbase membership information to correct the inconsistent age and region data in merch

NOTE: the age bracket in merch table is ESTIMATED, therefore, this age must be corrected with fanbase

In [19]:
merged_df = merged_df.dropna()

In [20]:
diffs = merged_df[merged_df['Customer_Age_Group_merch'] != merged_df['Customer_Age_Group_fanbase']]
print(f"{len(diffs)} rows have mismatched age groups.")

29913 rows have mismatched age groups.


In [21]:
merged_df['Customer_Age_Group_merch'] = merged_df['Customer_Age_Group_fanbase']


more inconsistencies with customer region - international vs domestic not matching up. unlike age group, it is not made clear if one is more reliable than the other (fanbase vs merch).
Strategy: if duplcates are consistent, leave it alone. if values differ between duplicates, categorize as domestic or international based on fanbase value - more reliable measure due to temporary nature of merch purchases vs registering an address for fanbase purchase (could be bought as a gift for someone else)

Set the merch customer region to be domestic if the product was bought in the team store

In [22]:
region_mismatches = merged_df[merged_df['Customer_Region_merch'] != merged_df['Customer_Region_fanbase']]
print(f"{len(region_mismatches)} region mismatches")

41710 region mismatches


In [23]:
merged_df['Customer_Region_merch'] = np.where(merged_df['Channel'] == 'Team Store', 'Domestic', merged_df['Customer_Region_merch'])


In [24]:
conflict_ids = (
    merged_df.groupby('Member_ID')['Customer_Region_merch']
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

In [25]:
conflict_mask = merged_df['Member_ID'].isin(conflict_ids)

# Apply conditional overwrite
merged_df.loc[conflict_mask, 'Customer_Region_merch'] = merged_df.loc[conflict_mask, 'Customer_Region_fanbase'].apply(
    lambda x: 'Domestic' if x == 'Canada' else 'International'
)

To sum up the rules for region:
1. if purchased in the team store, classify as customer merch region as domestic
2. if the customer merch region is consistent between duplicates, leave it alone
3. if the customer merch region is inconsistent between duplicates, default to membership purchase country

In [26]:
merch = merch.merge(
    merged_df[['Member_ID', 'Customer_Region_merch', 'Customer_Age_Group_merch']],
    on='Member_ID',
    how='left'
)

In [27]:
merch['Customer_Region_original'] = merch['Customer_Region']
merch['Customer_Age_Group_original'] = merch['Customer_Age_Group']
merch['Customer_Region'] = merch['Customer_Region_merch'].combine_first(merch['Customer_Region'])
merch['Customer_Age_Group'] = merch['Customer_Age_Group_merch'].combine_first(merch['Customer_Age_Group'])

In [28]:
merch.drop(columns=['Customer_Region_merch', 'Customer_Age_Group_merch'], inplace=True)

In [29]:
merch.to_csv('data/processed/cleaned_merch.csv', index = False)

In [30]:
merged_df = merged_df.drop_duplicates()

merged_df

,Product_ID,Barcode,Item_Category,Item_Name,Size,Unit_Price,Customer_Age_Group_merch,Customer_Region_merch,Promotion,Channel,Selling_Date,Member_ID,Arrival_Date,Customer_Age_Group_fanbase,Games_Attended,Seasonal_Pass,Customer_Region_fanbase,_merge
2403,10000004.0,20053292.0,Jersey,Away Jersey,L,152.0,41–60,Domestic,True,Online,2023-12-01 03:15:00,10000000,2023-12-10 09:03:00,41–60,5,False,Canada,both
2404,10000003.0,20012491.0,Hoodie,Training Hoodie,M,75.0,18–25,Domestic,False,Online,2023-12-01 05:45:00,10000001,2023-12-08 10:20:00,18–25,6,False,Canada,both
2405,10000014.0,20052572.0,Youth Hoodie,Youth Training Hoodie,YL,50.0,41–60,Domestic,True,Online,2023-12-01 01:35:00,10000002,2023-12-08 15:18:00,41–60,5,False,Canada,both
2407,10000002.0,20066583.0,Hoodie,Team Hoodie,S,75.0,18–25,Domestic,True,Online,2023-12-01 02:29:00,10000003,2023-12-10 16:58:00,18–25,2,False,Canada,both
2409,10000002.0,20067125.0,Hoodie,Team Hoodie,XXL,75.0,18–25,Domestic,False,Online,2023-12-01 12:53:00,10000004,2023-12-09 15:00:00,18–25,8,False,India,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70272,10000004.0,20015845.0,Jersey,Away Jersey,M,152.0,18–25,International,False,Online,2024-12-31 01:55:00,10044513,2025-01-08 11:06:00,18–25,4,False,Canada,both
70274,10000003.0,20031797.0,Hoodie,Training Hoodie,M,75.0,26–40,Domestic,False,Online,2024-12-31 16:23:00,10044515,2025-01-06 08:18:00,26–40,7,False,USA,both
70276,10000006.0,20017507.0,Jersey,Retro Jersey,XXL,152.0,18–25,Domestic,True,Online,2024-12-31 14:40:00,10044516,2025-01-06 12:37:00,18–25,6,False,Canada,both
70277,10000004.0,20036110.0,Jersey,Away Jersey,L,152.0,18–25,Domestic,True,Online,2024-12-31 01:32:00,10044517,2025-01-09 15:33:00,18–25,4,False,Canada,both


note that this merged file has all of the merch data, but only the fanbase info beloning to fans who purchased merch

In [31]:
merged_df = pd.merge(
    merch,
    fanbase,
    on='Member_ID',
    how='outer',
    suffixes=('_merch', '_fanbase')
)

merged_df['Customer_Age_Group'] = merged_df['Customer_Age_Group_merch'].combine_first(merged_df['Customer_Age_Group_fanbase'])
merged_df['Customer_Region'] = merged_df['Customer_Region_merch'].combine_first(merged_df['Customer_Region_fanbase'])

merged_df.drop(columns=['Customer_Age_Group_merch', 'Customer_Age_Group_fanbase', 
                        'Customer_Region_merch', 'Customer_Region_fanbase', 
                        'Customer_Region_original', 'Customer_Age_Group_original'], inplace=True)




In [32]:
merged_df['Attended_Event'] = merged_df['Games_Attended'].notna() & (merged_df['Games_Attended'] > 0)
merged_df['Purchased_Merch'] = merged_df['Item_Category'].notna()

merged_df

,Product_ID,Barcode,Item_Category,Item_Name,Size,Unit_Price,Promotion,Channel,Selling_Date,Member_ID,Arrival_Date,Games_Attended,Seasonal_Pass,Customer_Age_Group,Customer_Region,Attended_Event,Purchased_Merch
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,1370,NaT,4,False,26–40,Canada,True,False
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,7913,NaT,7,False,26–40,Canada,True,False
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,9041,NaT,4,False,18–25,Canada,True,False
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,13517,NaT,4,False,60+,Canada,True,False
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,18474,NaT,4,False,<18,India,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111526,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,99977604,NaT,1,False,26–40,Canada,True,False
111527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,99986747,NaT,7,False,18–25,Canada,True,False
111528,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,99991696,NaT,1,False,<18,Canada,True,False
111529,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,99995269,NaT,6,False,41–60,Canada,True,False


In [33]:
merged_df.to_csv('data/processed/merch_fanbase_merged.csv', index=False)

## Ensure Stadium has analysis-ready datatypes

convert to appropriate dtypes, feature engineer a revenue flag col (for later revenue based analytics)

- `Revenue_Flag`: Categorizes revenue as Gain or Loss.

In [34]:

stadium['Source'] = stadium['Source'].str.strip()
stadium['Revenue_Flag'] = stadium['Revenue'].apply(lambda x: 'Loss' if x < 0 else 'Zero' if x == 0 else 'Gain')


In [35]:
stadium

,Month,Source,Revenue,Revenue_Flag
0,1,Upper Bowl,25020,Gain
1,2,Upper Bowl,686664,Gain
2,3,Upper Bowl,434112,Gain
3,4,Upper Bowl,390879,Gain
4,5,Upper Bowl,286524,Gain
...,...,...,...,...
139,8,Advertising,416600,Gain
140,9,Advertising,382300,Gain
141,10,Advertising,421650,Gain
142,11,Advertising,416100,Gain


In [36]:
stadium.to_csv('data/processed/clean_stadium.csv', index = False)

Also feature engineer for fanbase

- `Region_Type`: Domestic vs International based on fanbase country.

In [37]:
fanbase['Region_Type'] = fanbase['Customer_Region'].apply(lambda x: 'Domestic' if x == 'Canada' else 'International')

In [38]:
fanbase.to_csv('data/processed/fanbase_clean.csv', index = False)